In [ ]:
import pathlib
import numpy as np
import pandas as pd
import pyFLUT

# Generate reference CSV for pdf_flut.h5

Loads `pdf_flut.h5`, samples a set of test points, computes reference lookup results
and normalised dimension values, then writes `data/results_pdf_flut.csv`.

The CSV is consumed by the C++ Catch2 tests via `compareCSVNormalization` and
`compareCSVLookup` in `test_lookup_table_csv.cpp`.

Column layout (matches what `compareCSVNormalization` expects in dimension order):
- `Z`         – raw mixture-fraction input (dim 0, not normalised)
- `Zvar`      – physical variance = `Zvar_norm * Z * (1 - Z)` (dim 1 physical value)
- `hMean`     – raw enthalpy input read from table output (dim 2 physical value)
- `yc`        – raw progress-variable input read from table output (dim 3 physical value)
- `Zvar_norm` – normalised variance on the Zvar_norm grid [0, 0.1] (dim 1 normalised)
- `Hnorm`     – normalised enthalpy on the Hnorm grid [0, 1] (dim 2 normalised)
- `cc`        – normalised progress variable on the cc grid [0, 1] (dim 3 normalised)
- output variable columns (T, rho, …)

Note: pyFLUT works entirely in normalised space `(Z, Zvar_norm, Hnorm, cc)`.
Physical `Zvar` is not a table input — it is reverse-engineered as
`Zvar = Zvar_norm * Z * (1 - Z)` (inverse of `normalizeVariance` in `lookup_table.cpp`).

In [ ]:
repo_root = pathlib.Path(".").resolve().parent
flut_path = repo_root / "data" / "pdf_flut.h5"
csv_path  = repo_root / "data" / "results_pdf_flut.csv"

print("FLUT:", flut_path)
print("CSV: ", csv_path)

In [ ]:
flut = pyFLUT.read_h5(str(flut_path))
print("Input variables: ", list(flut.input_dict.keys()))
print("Output variables:", flut.output_variables[:10], "...")

## Sample test points

Sample directly in normalised space `(Z, Zvar_norm, Hnorm, cc)` — this is the
coordinate system pyFLUT uses. A few values per axis keeps the CSV manageable
while covering interior points and boundary behaviour.

In [ ]:
Z_vals        = np.array([0.02, 0.1, 0.3, 0.5, 0.7, 0.9])
Zvar_norm_vals = np.array([0.0, 0.01, 0.05, 0.1])   # matches the Zvar_norm grid exactly
Hnorm_vals    = np.array([0.05, 0.25, 0.5, 0.75, 1.0])
cc_vals       = np.array([0.0, 0.25, 0.5, 0.75, 1.0])

Z_g, Zvar_norm_g, Hnorm_g, cc_g = np.meshgrid(
    Z_vals, Zvar_norm_vals, Hnorm_vals, cc_vals, indexing="ij"
)

Z_flat         = Z_g.ravel()
Zvar_norm_flat = Zvar_norm_g.ravel()
Hnorm_flat     = Hnorm_g.ravel()
cc_flat        = cc_g.ravel()

print(f"{len(Z_flat)} test points")

## Lookup and collect normalised dimension values

In [ ]:
output_vars = ["T", "rho", "hMean", "yc", "omega_yc", "CO2", "CO", "H2O", "OH", "N2"]

rows = []
for Z, Zvar_norm, Hnorm, cc in zip(Z_flat, Zvar_norm_flat, Hnorm_flat, cc_flat):
    kwargs = {"Z": Z, "Zvar_norm": Zvar_norm, "Hnorm": Hnorm, "cc": cc}

    # Physical Zvar: inverse of normalizeVariance() in lookup_table.cpp
    Zvar = Zvar_norm * Z * (1.0 - Z)

    row = {
        # Physical inputs (what the C++ library receives)
        "Z":     Z,
        "Zvar":  Zvar,
        "hMean": flut.extract_values("hMean", **kwargs),
        "yc":    flut.extract_values("yc",    **kwargs),
        # Normalised dimension values (what compareCSVNormalization checks)
        "Zvar_norm": Zvar_norm,
        "Hnorm":     Hnorm,
        "cc":        cc,
    }
    for var in output_vars:
        row[var] = flut.extract_values(var, **kwargs)
    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head()

## Write CSV and return path

In [ ]:
df.to_csv(csv_path, index=False)
print(csv_path)